In [2]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import json

In [3]:
# qqp_02.ipynb — Análisis nacional QQP
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sina.db.repository import QQPRepository

# Cargar TODOS los datos de la tabla
repo = QQPRepository()

# Necesitamos un método que jale todo, no solo por municipio
# Opción A: query directo con SQLAlchemy
from sqlalchemy import select
from sina.db.models import PrecioQQP

with repo.Session() as session:
    stmt = select(PrecioQQP)
    rows = session.execute(stmt).scalars().all()
    data = [
        {
            "producto":         r.producto,
            "presentacion":     r.presentacion,
            "marca":            r.marca,
            "categoria":        r.categoria,
            "precio":           r.precio,
            "fecha_registro":   r.fecha_registro,
            "cadena_comercial": r.cadena_comercial,
            "nombre_comercial": r.nombre_comercial,
            "direccion":        r.direccion,
            "estado":           r.estado,
            "municipio":        r.municipio,
            "latitud":          r.latitud,
            "longitud":         r.longitud,
        }
        for r in rows
    ]

df = pd.DataFrame(data)
df['precio'] = df['precio'].astype(float)
print(f"Total registros: {len(df):,}")
print(f"Estados: {df['estado'].nunique()}")
print(f"Municipios: {df['municipio'].nunique()}")
print(f"Combinaciones estado-municipio: {df.groupby(['estado','municipio']).ngroups}")

🗄️  Usando SQLite local: C:\Users\PANDA\Documents\GitHub\sina\datos\db\sina_data.db
Total registros: 549,936
Estados: 30
Municipios: 70
Combinaciones estado-municipio: 72


In [35]:


# Canasta V3
CANASTA_V3 = {
    'Aceite':           ['Aceite'],
    'Arroz':            ['Arroz'],
    'Frijol':           ['Frijol', 'Frijoles'],
    'Azúcar':           ['Azúcar'],
    'Huevo':            ['Huevo'],
    'Leche':            ['Leche Ultrapasteurizada', 'Leche Condensada', 
                         'Leche Evaporada', 'Leche en Polvo'],
    'Pan':              ['Pan Blanco Bolillo', 'Pan de Caja', 
                         'Pan Dulce', 'Pastellios y Pan Dulce Empaquetado'],
    'Tortilla':         ['Tortilla de Maíz'],
    'Pollo':            ['Carne Pollo'],
    'Atún':             ['Atún'],
    'Jabón de pasta':   ['Jabón de Pasta'],
    'Papel higiénico':  ['Papel Higiénico'],
    'Pasta para sopa':  ['Pasta para Sopa'],
    'Sal':              ['Sal Molida de Mesa'],
}

CADENAS_SUPER = [
    'Wal-mart', 'Bodega Aurrera', 'Hipermercado Soriana',
    'Chedraui', 'Mega Soriana', 'Wal-mart Express',
    'Ley', 'Soriana Super', 'H.e.b.', 'Mercado Soriana',
    'La Comer', 'Minisuper', 'Fresko la Comer',
    'Bodega Aurrera Express', 'Chedraui Selecto',
    'Super Chedraui', 'S Mart', 'Sumesa', 'Alsuper',
    'Soriana Express', 'Superissste', 'City Market',
    'Mercado Publico', 'Central de Abastos',
]

df_super = df[df['cadena_comercial'].isin(CADENAS_SUPER)].copy()

In [36]:
def deduplicar_opciones(df_local):
    """
    Elimina duplicados: mismo producto+presentacion+marca+cadena+precio
    Mantiene solo 1 registro por combinación única
    """
    return df_local.drop_duplicates(
        subset=['producto', 'presentacion', 'marca', 'cadena_comercial', 'precio']
    )

In [37]:
def obtener_datos_municipio(df, estado, municipio, canasta):
    """
    Retorna un DataFrame limpio con todos los productos de canasta
    para un estado-municipio, deduplicado y enriquecido.
    """
    mask_loc = (df['estado'] == estado) & (df['municipio'] == municipio)
    df_local = df[mask_loc].copy()
    
    resultados = []
    for item_name, productos in canasta.items():
        mask = df_local['producto'].isin(productos)
        subset = df_local[mask].copy()
        if len(subset) == 0:
            continue
        subset['item_canasta'] = item_name
        resultados.append(subset)
    
    if not resultados:
        return pd.DataFrame()
    
    df_canasta = pd.concat(resultados, ignore_index=True)
    df_canasta = deduplicar_opciones(df_canasta)
    
    return df_canasta

# Test
df_hmo = obtener_datos_municipio(df_super, 'Sonora', 'Hermosillo', CANASTA_V3)
print(f"Hermosillo: {len(df_hmo)} registros únicos, {df_hmo['item_canasta'].nunique()} items")

Hermosillo: 475 registros únicos, 14 items


In [38]:
def grafica_panorama_canasta(df_muni, estado, municipio):
    """
    Box plot horizontal: rango de precios por item de canasta.
    El usuario ve de un vistazo qué tan disperso es cada producto.
    """
    # Ordenar items por precio mediano
    orden = (
        df_muni.groupby('item_canasta')['precio']
        .median()
        .sort_values()
        .index.tolist()
    )
    
    fig = px.box(
        df_muni,
        x='precio',
        y='item_canasta',
        color='item_canasta',
        orientation='h',
        category_orders={'item_canasta': orden},
        hover_data=['marca', 'presentacion', 'cadena_comercial'],
        title=f'Canasta básica — {municipio}, {estado}<br>'
              f'<sub>Distribución de precios por producto (todas las presentaciones)</sub>',
    )
    
    fig.update_layout(
        showlegend=False,
        height=500,
        xaxis_title='Precio ($)',
        yaxis_title='',
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
    )
    
    # Agregar línea vertical con costo mínimo total
    costo_min = df_muni.groupby('item_canasta')['precio'].min().sum()
    fig.add_annotation(
        text=f'Canasta mínima: ${costo_min:,.0f}',
        xref='paper', yref='paper',
        x=0.98, y=1.05,
        showarrow=False,
        font=dict(size=14, color='#2ecc71'),
    )
    
    fig.show()

grafica_panorama_canasta(df_hmo, 'Sonora', 'Hermosillo')

In [39]:
def grafica_heatmap_cadenas(df_muni, estado, municipio):
    """
    Heatmap: items × cadenas, mostrando el precio más barato 
    que cada cadena ofrece para cada item.
    """
    # Precio mínimo por item × cadena
    pivot = (
        df_muni
        .groupby(['item_canasta', 'cadena_comercial'])['precio']
        .min()
        .reset_index()
        .pivot_table(index='item_canasta', columns='cadena_comercial', values='precio')
    )
    
    # Normalizar por fila para colores (1.0 = más barato)
    pivot_norm = pivot.div(pivot.min(axis=1), axis=0)
    
    fig = go.Figure(data=go.Heatmap(
        z=pivot_norm.values,
        x=pivot.columns.tolist(),
        y=pivot.index.tolist(),
        text=pivot.values,
        texttemplate='$%{text:.0f}',
        textfont=dict(size=11),
        colorscale=[
            [0.0, '#2ecc71'],   # verde = barato
            [0.3, '#f1c40f'],   # amarillo
            [1.0, '#e74c3c'],   # rojo = caro
        ],
        zmin=1.0,
        zmax=1.5,
        colorbar=dict(
            title='Ratio vs<br>más barato',
            tickvals=[1.0, 1.25, 1.5],
            ticktext=['1.0x (mejor)', '1.25x', '1.5x+'],
        ),
        hovertemplate=(
            '<b>%{y}</b><br>'
            '%{x}: $%{text:.2f}<br>'
            'Ratio: %{z:.2f}x vs mejor precio'
            '<extra></extra>'
        ),
    ))
    
    fig.update_layout(
        title=f'Comparativo por cadena — {municipio}, {estado}<br>'
              f'<sub>Precio más barato por producto × cadena (verde = mejor precio)</sub>',
        height=500,
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
        xaxis=dict(side='bottom'),
    )
    
    fig.show()

grafica_heatmap_cadenas(df_hmo, 'Sonora', 'Hermosillo')

In [40]:
def grafica_detalle_item(df_muni, item_name, estado, municipio):
    """
    Strip plot: cada punto es una opción (marca×cadena).
    Faceteado por presentación.
    El usuario ve TODAS las opciones disponibles.
    """
    df_item = df_muni[df_muni['item_canasta'] == item_name].copy()
    
    if len(df_item) == 0:
        print(f"Sin datos para {item_name}")
        return
    
    # Ordenar presentaciones por precio mediano
    orden_pres = (
        df_item.groupby('presentacion')['precio']
        .median()
        .sort_values()
        .index.tolist()
    )
    
    fig = px.strip(
        df_item,
        x='precio',
        y='presentacion',
        color='cadena_comercial',
        category_orders={'presentacion': orden_pres},
        hover_data=['marca', 'nombre_comercial', 'precio'],
        title=f'{item_name} — {municipio}, {estado}<br>'
              f'<sub>{len(df_item)} opciones en {df_item["cadena_comercial"].nunique()} cadenas '
              f'y {df_item["presentacion"].nunique()} presentaciones</sub>',
    )
    
    fig.update_layout(
        height=max(400, len(orden_pres) * 50 + 200),
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
        xaxis_title='Precio ($)',
        yaxis_title='',
        legend_title='Cadena',
    )
    
    fig.show()

grafica_detalle_item(df_hmo, 'Aceite', 'Sonora', 'Hermosillo')
grafica_detalle_item(df_hmo, 'Huevo', 'Sonora', 'Hermosillo')
grafica_detalle_item(df_hmo, 'Leche', 'Sonora', 'Hermosillo')

In [41]:
def grafica_canasta_optima(df_muni, estado, municipio):
    """
    Waterfall: construye la canasta más barata item por item.
    Muestra de dónde comprar cada cosa.
    """
    optimos = []
    for item in sorted(df_muni['item_canasta'].unique()):
        sub = df_muni[df_muni['item_canasta'] == item]
        mejor = sub.nsmallest(1, 'precio').iloc[0]
        optimos.append({
            'item': item,
            'precio': mejor['precio'],
            'cadena': mejor['cadena_comercial'],
            'marca': mejor['marca'],
            'presentacion': mejor['presentacion'],
        })
    
    df_opt = pd.DataFrame(optimos).sort_values('precio', ascending=True)
    total = df_opt['precio'].sum()
    
    # Colores por cadena
    cadenas_unicas = df_opt['cadena'].unique()
    colores_cadena = px.colors.qualitative.Set2[:len(cadenas_unicas)]
    color_map = dict(zip(cadenas_unicas, colores_cadena))
    
    fig = go.Figure(go.Waterfall(
        name='Canasta',
        orientation='v',
        measure=['relative'] * len(df_opt) + ['total'],
        x=df_opt['item'].tolist() + ['TOTAL'],
        y=df_opt['precio'].tolist() + [0],
        text=[f"${p:.0f}<br>{c}" for p, c in zip(df_opt['precio'], df_opt['cadena'])] + [f"${total:.0f}"],
        textposition='outside',
        connector=dict(line=dict(color='rgba(0,0,0,0)')),
        increasing=dict(marker=dict(color='#2ecc71')),
        totals=dict(marker=dict(color='#3498db')),
        hovertemplate=(
            '<b>%{x}</b><br>'
            'Precio: $%{y:.2f}<br>'
            '<extra></extra>'
        ),
    ))
    
    fig.update_layout(
        title=f'Canasta óptima — {municipio}, {estado}<br>'
              f'<sub>El producto más barato de cada categoría (cualquier presentación)</sub>',
        height=500,
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
        yaxis_title='Precio ($)',
        showlegend=False,
    )
    
    fig.show()

grafica_canasta_optima(df_hmo, 'Sonora', 'Hermosillo')

In [42]:
def grafica_ranking_cadenas(df_muni, estado, municipio):
    """
    Para cada cadena, el costo total de SU canasta más barata.
    Barras agrupadas con desglose por item.
    """
    # Precio mínimo por item × cadena
    min_por_cadena = (
        df_muni
        .groupby(['cadena_comercial', 'item_canasta'])['precio']
        .min()
        .reset_index()
    )
    
    # Total por cadena (solo items que la cadena tenga)
    totales = (
        min_por_cadena
        .groupby('cadena_comercial')
        .agg(
            costo_total=('precio', 'sum'),
            n_items=('item_canasta', 'nunique')
        )
        .sort_values('costo_total')
        .reset_index()
    )
    
    n_items_total = df_muni['item_canasta'].nunique()
    
    fig = px.bar(
        min_por_cadena,
        x='cadena_comercial',
        y='precio',
        color='item_canasta',
        category_orders={
            'cadena_comercial': totales['cadena_comercial'].tolist()
        },
        title=f'Costo de canasta por cadena — {municipio}, {estado}<br>'
              f'<sub>Producto más barato de cada categoría por cadena</sub>',
        hover_data=['item_canasta', 'precio'],
    )
    
    # Agregar anotación de total
    for _, row in totales.iterrows():
        fig.add_annotation(
            x=row['cadena_comercial'],
            y=row['costo_total'],
            text=f"${row['costo_total']:.0f}<br>({row['n_items']}/{n_items_total})",
            showarrow=False,
            yshift=15,
            font=dict(size=10, color='#333'),
        )
    
    fig.update_layout(
        height=500,
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
        yaxis_title='Costo ($)',
        xaxis_title='',
        barmode='stack',
        legend_title='Producto',
    )
    
    fig.show()

grafica_ranking_cadenas(df_hmo, 'Sonora', 'Hermosillo')

In [43]:
def grafica_ahorro(df_muni, estado, municipio):
    """
    Lollipop chart: para cada item, punto del más barato 
    y punto del más caro. La línea entre ellos = ahorro potencial.
    """
    resumen = []
    for item in df_muni['item_canasta'].unique():
        sub = df_muni[df_muni['item_canasta'] == item]
        mejor = sub.nsmallest(1, 'precio').iloc[0]
        peor = sub.nlargest(1, 'precio').iloc[0]
        resumen.append({
            'item': item,
            'precio_min': mejor['precio'],
            'cadena_min': mejor['cadena_comercial'],
            'pres_min': mejor['presentacion'],
            'precio_max': peor['precio'],
            'cadena_max': peor['cadena_comercial'],
            'pres_max': peor['presentacion'],
            'ahorro': peor['precio'] - mejor['precio'],
            'ahorro_pct': (peor['precio'] - mejor['precio']) / peor['precio'] * 100,
        })
    
    df_res = pd.DataFrame(resumen).sort_values('ahorro_pct', ascending=True)
    
    fig = go.Figure()
    
    for _, row in df_res.iterrows():
        # Línea entre min y max
        fig.add_trace(go.Scatter(
            x=[row['precio_min'], row['precio_max']],
            y=[row['item'], row['item']],
            mode='lines',
            line=dict(color='#bdc3c7', width=3),
            showlegend=False,
            hoverinfo='skip',
        ))
        
        # Punto más barato (verde)
        fig.add_trace(go.Scatter(
            x=[row['precio_min']],
            y=[row['item']],
            mode='markers+text',
            marker=dict(color='#2ecc71', size=12),
            text=[f"${row['precio_min']:.0f}"],
            textposition='middle left',
            name='Más barato',
            showlegend=False,
            hovertemplate=(
                f"<b>{row['item']}</b><br>"
                f"${row['precio_min']:.2f} — {row['cadena_min']}<br>"
                f"{row['pres_min']}"
                f"<extra></extra>"
            ),
        ))
        
        # Punto más caro (rojo)
        fig.add_trace(go.Scatter(
            x=[row['precio_max']],
            y=[row['item']],
            mode='markers+text',
            marker=dict(color='#e74c3c', size=12),
            text=[f"${row['precio_max']:.0f}"],
            textposition='middle right',
            name='Más caro',
            showlegend=False,
            hovertemplate=(
                f"<b>{row['item']}</b><br>"
                f"${row['precio_max']:.2f} — {row['cadena_max']}<br>"
                f"{row['pres_max']}"
                f"<extra></extra>"
            ),
        ))
    
    fig.update_layout(
        title=f'Rango de precios — {municipio}, {estado}<br>'
              f'<sub>🟢 Más barato vs 🔴 Más caro por categoría</sub>',
        height=500,
        template='plotly_white',
        font=dict(family='Inter, sans-serif'),
        xaxis_title='Precio ($)',
    )
    
    fig.show()

grafica_ahorro(df_hmo, 'Sonora', 'Hermosillo')

In [44]:
# Municipio grande
grafica_panorama_canasta(
    obtener_datos_municipio(df_super, 'Ciudad de Mexico', 'Benito Juárez', CANASTA_V3),
    'Ciudad de Mexico', 'Benito Juárez'
)

# Municipio mediano
grafica_panorama_canasta(
    obtener_datos_municipio(df_super, 'Sonora', 'Hermosillo', CANASTA_V3),
    'Sonora', 'Hermosillo'
)

# Municipio chico
grafica_panorama_canasta(
    obtener_datos_municipio(df_super, 'Guerrero', 'Acapulco de Juárez', CANASTA_V3),
    'Guerrero', 'Acapulco de Juárez'
)

In [55]:
# qqp_03.ipynb — Prototipo visual limpio

import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ═══════════════════════════════════════════════════════
# PALETA Y CONFIGURACIÓN GLOBAL
# ═══════════════════════════════════════════════════════

# Paleta principal: teal monocromático (inspirada en SonoraEnDatos)
COLORES = {
    'primario':     '#0d9488',  # teal-600
    'primario_osc': '#0f766e',  # teal-700
    'primario_cla': '#5eead4',  # teal-300
    'fondo_barra':  '#ccfbf1',  # teal-100
    'acento':       '#f97316',  # naranja (para destacar 1 cosa)
    'negativo':     '#ef4444',  # rojo suave (solo para "caro")
    'texto':        '#1e293b',  # slate-800
    'texto_sec':    '#64748b',  # slate-500
    'linea':        '#e2e8f0',  # slate-200
    'fondo':        '#ffffff',
}

# Degradado para barras (de oscuro a claro, mismo tono)
def degradado_teal(n):
    """Genera n colores del más oscuro al más claro en teal"""
    base_colors = [
        '#134e4a', '#115e59', '#0f766e', '#0d9488', 
        '#14b8a6', '#2dd4bf', '#5eead4', '#99f6e4', '#ccfbf1'
    ]
    if n <= len(base_colors):
        step = max(1, len(base_colors) // n)
        return [base_colors[i * step] for i in range(n)]
    return [COLORES['primario']] * n

# Layout base que se aplica a TODAS las gráficas
LAYOUT_BASE = dict(
    font=dict(
        family='Inter, -apple-system, sans-serif',
        color=COLORES['texto'],
        size=13,
    ),
    paper_bgcolor=COLORES['fondo'],
    plot_bgcolor=COLORES['fondo'],
    margin=dict(l=20, r=20, t=80, b=20),
    title=dict(
        font=dict(size=18, color=COLORES['texto']),
        x=0.02,
        xanchor='left',
    ),
    xaxis=dict(
        gridcolor=COLORES['linea'],
        gridwidth=0.5,
        zeroline=False,
        tickfont=dict(size=11, color=COLORES['texto_sec']),
    ),
    yaxis=dict(
        gridcolor=COLORES['linea'],
        gridwidth=0.5,
        zeroline=False,
        tickfont=dict(size=11, color=COLORES['texto_sec']),
    ),
    hoverlabel=dict(
        bgcolor='white',
        font_size=12,
        font_family='Inter, sans-serif',
        bordercolor=COLORES['linea'],
    ),
    showlegend=False,
)

In [56]:
def grafica_ranking_cadenas(df_muni, estado, municipio):
    # Precio mínimo por item × cadena
    min_por_cadena = (
        df_muni
        .groupby(['cadena_comercial', 'item_canasta'])['precio']
        .min()
        .reset_index()
    )
    
    totales = (
        min_por_cadena
        .groupby('cadena_comercial')
        .agg(costo_total=('precio', 'sum'), n_items=('item_canasta', 'nunique'))
        .sort_values('costo_total', ascending=True)
        .reset_index()
    )
    
    n_items_total = df_muni['item_canasta'].nunique()
    totales = totales[totales['n_items'] >= n_items_total * 0.5]
    
    n = len(totales)
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=totales['costo_total'],
        y=totales['cadena_comercial'],
        orientation='h',
        marker=dict(
            color=degradado_teal(n)[::-1],
            line=dict(width=0),
        ),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Canasta: $%{x:,.0f}<br>'
            '<extra></extra>'
        ),
    ))
    
    # Anotaciones
    for i, row in totales.iterrows():
        fig.add_annotation(
            x=row['costo_total'],
            y=row['cadena_comercial'],
            text=f"  ${row['costo_total']:,.0f}",
            showarrow=False,
            xanchor='left',
            font=dict(
                size=14,
                color=COLORES['texto'],
                weight='bold' if i == 0 else 'normal',
            ),
        )
    
    # 🔑 MERGE limpio del layout
    layout = LAYOUT_BASE.copy()
    
    layout['title'] = dict(
        text=(
            f'¿Dónde sale más barata la canasta básica?<br>'
            f'<span style="font-size:13px;color:{COLORES["texto_sec"]}">'
            f'{municipio}, {estado} · Producto más barato por categoría</span>'
        ),
    )
    
    layout['height'] = max(300, n * 55 + 120)
    
    layout['xaxis'] = {
        **LAYOUT_BASE.get('xaxis', {}),
        'title': '',
        'showticklabels': False,
        'showgrid': False,
    }
    
    layout['yaxis'] = {
        **LAYOUT_BASE.get('yaxis', {}),
        'title': '',
        'tickfont': dict(size=13, color=COLORES['texto']),
    }
    
    fig.update_layout(**layout)
    
    fig.show()


grafica_ranking_cadenas(df_hmo, 'Sonora', 'Hermosillo')

In [47]:
def grafica_rango_precios(df_muni, estado, municipio):
    """
    Dumbbell: min vs max por item.
    Sin boxplot. Solo dos puntos y una línea.
    Un color para barato, otro para caro.
    """
    resumen = []
    for item in df_muni['item_canasta'].unique():
        sub = df_muni[df_muni['item_canasta'] == item]
        mejor = sub.nsmallest(1, 'precio').iloc[0]
        peor = sub.nlargest(1, 'precio').iloc[0]
        resumen.append({
            'item': item,
            'precio_min': mejor['precio'],
            'cadena_min': mejor['cadena_comercial'],
            'pres_min': mejor['presentacion'],
            'marca_min': mejor['marca'],
            'precio_max': peor['precio'],
            'cadena_max': peor['cadena_comercial'],
            'pres_max': peor['presentacion'],
        })
    
    df_res = pd.DataFrame(resumen).sort_values('precio_min', ascending=True)
    
    fig = go.Figure()
    
    for _, row in df_res.iterrows():
        # Línea conectora (gris suave)
        fig.add_trace(go.Scatter(
            x=[row['precio_min'], row['precio_max']],
            y=[row['item'], row['item']],
            mode='lines',
            line=dict(color=COLORES['linea'], width=2),
            showlegend=False,
            hoverinfo='skip',
        ))
        
        # Punto más barato (teal)
        fig.add_trace(go.Scatter(
            x=[row['precio_min']],
            y=[row['item']],
            mode='markers',
            marker=dict(color=COLORES['primario'], size=10),
            showlegend=False,
            hovertemplate=(
                f"<b>{row['item']}</b> — Más barato<br>"
                f"${row['precio_min']:.2f}<br>"
                f"{row['marca_min']} · {row['pres_min']}<br>"
                f"{row['cadena_min']}"
                f"<extra></extra>"
            ),
        ))
        
        # Punto más caro (naranja/acento)
        fig.add_trace(go.Scatter(
            x=[row['precio_max']],
            y=[row['item']],
            mode='markers',
            marker=dict(color=COLORES['acento'], size=10, opacity=0.7),
            showlegend=False,
            hovertemplate=(
                f"<b>{row['item']}</b> — Más caro<br>"
                f"${row['precio_max']:.2f}<br>"
                f"{row['pres_max']}<br>"
                f"{row['cadena_max']}"
                f"<extra></extra>"
            ),
        ))
        
        # Anotación del precio mínimo
        fig.add_annotation(
            x=row['precio_min'], y=row['item'],
            text=f"${row['precio_min']:.0f}",
            showarrow=False, xanchor='right', xshift=-8,
            font=dict(size=11, color=COLORES['primario']),
        )
        
        # Anotación del precio máximo
        fig.add_annotation(
            x=row['precio_max'], y=row['item'],
            text=f"${row['precio_max']:.0f}",
            showarrow=False, xanchor='left', xshift=8,
            font=dict(size=11, color=COLORES['acento']),
        )
    
    # Leyenda manual arriba
    fig.add_annotation(
        xref='paper', yref='paper', x=0.0, y=1.08,
        text=(
            f'<span style="color:{COLORES["primario"]}">●</span> Más barato  '
            f'<span style="color:{COLORES["acento"]}">●</span> Más caro'
        ),
        showarrow=False, font=dict(size=12),
    )
    
    layout = LAYOUT_BASE.copy()

    layout['title'] = dict(
        text=(
            f'Rango de precios por producto<br>'
            f'<span style="font-size:13px;color:{COLORES["texto_sec"]}">'
            f'{municipio}, {estado} · Todas las presentaciones y marcas</span>'
        ),
    )

    layout['height'] = max(400, len(df_res) * 38 + 120)

    layout['xaxis'] = {
        **LAYOUT_BASE.get('xaxis', {}),
        'title': 'Precio ($)',
        'showgrid': True,
    }

    layout['yaxis'] = {
        **LAYOUT_BASE.get('yaxis', {}),
        'title': '',
    }

    fig.update_layout(**layout)
    
    fig.show()

grafica_rango_precios(df_hmo, 'Sonora', 'Hermosillo')

In [57]:
def grafica_detalle_item(df_muni, item_name, estado, municipio):
    """
    Barras horizontales por presentación.
    Cada barra = precio más barato de esa presentación.
    Color degradado teal. Anotaciones con cadena y precio.
    """
    df_item = df_muni[df_muni['item_canasta'] == item_name].copy()
    
    if len(df_item) == 0:
        print(f"Sin datos para {item_name}")
        return
    
    # Precio mínimo por presentación
    mejor_por_pres = (
        df_item
        .sort_values('precio')
        .groupby('presentacion')
        .first()
        .reset_index()
        .sort_values('precio', ascending=True)
    )
    
    n = len(mejor_por_pres)
    colores = degradado_teal(n)[::-1]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=mejor_por_pres['precio'],
        y=mejor_por_pres['presentacion'],
        orientation='h',
        marker=dict(color=colores, line=dict(width=0)),
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Desde $%{x:,.2f}<br>'
            '<extra></extra>'
        ),
    ))
    
    # Anotaciones: precio + cadena
    for _, row in mejor_por_pres.iterrows():
        n_cadenas = df_item[df_item['presentacion'] == row['presentacion']]['cadena_comercial'].nunique()
        fig.add_annotation(
            x=row['precio'], y=row['presentacion'],
            text=f"  ${row['precio']:.0f} · {row['cadena_comercial']} ({n_cadenas} {'cadena' if n_cadenas == 1 else 'cadenas'})",
            showarrow=False, xanchor='left',
            font=dict(size=11, color=COLORES['texto_sec']),
        )
    
    layout = LAYOUT_BASE.copy()

    layout['title'] = dict(
        text=(
            f'{item_name}<br>'
            f'<span style="font-size:13px;color:{COLORES["texto_sec"]}">'
            f'{municipio}, {estado} · Precio más bajo por presentación</span>'
        ),
    )

    layout['height'] = max(300, n * 40 + 120)

    layout['xaxis'] = {
        **LAYOUT_BASE.get('xaxis', {}),
        'title': 'Precio ($)',
        'showgrid': True,
    }

    layout['yaxis'] = {
        **LAYOUT_BASE.get('yaxis', {}),
        'title': '',
        'tickfont': dict(size=11),
    }

    fig.update_layout(**layout)
    
    fig.show()



In [58]:
# Test con items de distinta complejidad
grafica_detalle_item(df_hmo, 'Aceite', 'Sonora', 'Hermosillo')


In [59]:
grafica_detalle_item(df_hmo, 'Huevo', 'Sonora', 'Hermosillo')

In [60]:
grafica_detalle_item(df_hmo, 'Leche', 'Sonora', 'Hermosillo')


In [61]:

grafica_detalle_item(df_hmo, 'Tortilla', 'Sonora', 'Hermosillo')

In [62]:
def grafica_comparativo_cadenas_item(df_muni, item_name, presentacion, estado, municipio):
    """
    Barras horizontales: precio del mismo producto en cada cadena.
    El usuario ya eligió qué quiere, solo compara DÓNDE comprarlo.
    """
    df_item = df_muni[
        (df_muni['item_canasta'] == item_name) & 
        (df_muni['presentacion'] == presentacion)
    ].copy()
    
    if len(df_item) == 0:
        print(f"Sin datos para {item_name} - {presentacion}")
        return
    
    # Precio mínimo por cadena (puede haber varias marcas)
    por_cadena = (
        df_item
        .sort_values('precio')
        .groupby('cadena_comercial')
        .first()
        .reset_index()
        .sort_values('precio', ascending=True)
    )
    
    n = len(por_cadena)
    colores = degradado_teal(n)[::-1]
    
    # Highlight: la más barata en teal oscuro, las demás más claras
    colores_barras = []
    for i in range(n):
        if i == 0:
            colores_barras.append(COLORES['primario_osc'])
        else:
            colores_barras.append(COLORES['primario_cla'])
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=por_cadena['precio'],
        y=por_cadena['cadena_comercial'],
        orientation='h',
        marker=dict(color=colores_barras, line=dict(width=0)),
        hovertemplate=(
            '<b>%{y}</b><br>'
            '$%{x:,.2f}<br>'
            '<extra></extra>'
        ),
    ))
    
    # Anotaciones
    precio_min = por_cadena['precio'].iloc[0]
    for _, row in por_cadena.iterrows():
        diferencia = row['precio'] - precio_min
        texto = f"  ${row['precio']:.2f}"
        if diferencia > 0:
            texto += f"  (+${diferencia:.2f})"
        
        fig.add_annotation(
            x=row['precio'], y=row['cadena_comercial'],
            text=texto,
            showarrow=False, xanchor='left',
            font=dict(
                size=12, 
                color=COLORES['primario_osc'] if diferencia == 0 else COLORES['texto_sec'],
                weight='bold' if diferencia == 0 else 'normal',
            ),
        )


    layout = LAYOUT_BASE.copy()

    layout['title'] = dict(
        text=(
            f'{item_name}<br>'
            f'<span style="font-size:13px;color:{COLORES["texto_sec"]}">'
            f'{municipio}, {estado} · Comparativo por cadena</span>'
        ),
    )

    layout['height'] = max(250, n * 45 + 120)

    layout['xaxis'] = {
        **LAYOUT_BASE.get('xaxis', {}),
        'title': '',
        'showticklabels': False,
        'showgrid': False,
    }

    layout['yaxis'] = {
        **LAYOUT_BASE.get('yaxis', {}),
        'title': '',
        'tickfont': dict(size=11),
    }

    fig.update_layout(**layout)
    
    fig.show()

# Ejemplo: Huevo C/18
grafica_comparativo_cadenas_item(df_hmo, 'Huevo', 'Paquete C/18 Blanco', 'Sonora', 'Hermosillo')
# Ejemplo: Aceite 850ml
grafica_comparativo_cadenas_item(df_hmo, 'Aceite', 'Botella 850 Ml. Vegetal', 'Sonora', 'Hermosillo')

In [63]:
def grafica_canasta_personalizada(seleccion, estado, municipio):
    """
    Waterfall con la canasta que el usuario armó.
    seleccion = lista de dicts: [{item, presentacion, precio, cadena, marca}]
    """
    df_sel = pd.DataFrame(seleccion).sort_values('precio')
    total = df_sel['precio'].sum()
    
    fig = go.Figure(go.Waterfall(
        orientation='v',
        measure=['relative'] * len(df_sel) + ['total'],
        x=[s['item'] for s in seleccion] + ['Total'],
        y=[s['precio'] for s in seleccion] + [0],
        text=[f"${s['precio']:.0f}" for s in seleccion] + [f"${total:.0f}"],
        textposition='outside',
        textfont=dict(size=12, color=COLORES['texto']),
        connector=dict(line=dict(color=COLORES['linea'], width=1)),
        increasing=dict(marker=dict(color=COLORES['primario'])),
        totals=dict(marker=dict(color=COLORES['primario_osc'])),
        hovertemplate=(
            '<b>%{x}</b><br>'
            '$%{y:,.2f}<br>'
            '<extra></extra>'
        ),
    ))


    layout = LAYOUT_BASE.copy()

    layout['title'] = dict(
        text=(
            f'Tu canasta: ${total:,.0f}<br>'
            f'<span style="font-size:13px;color:{COLORES["texto_sec"]}">'
            f'{municipio}, {estado} · Selección personalizada</span>'
        ),
    )

    layout['height'] = 400

    layout['xaxis'] = {
        **LAYOUT_BASE.get('xaxis', {}),
        'title': 'Precio ($)',
        'showgrid': False,
    }

    layout['yaxis'] = {
        **LAYOUT_BASE.get('yaxis', {}),
        'title': '',
        'tickfont': dict(size=11),
    }

    fig.update_layout(**layout)
    
    fig.show()

# Simular selección del usuario
seleccion_ejemplo = [
    {'item': 'Pan', 'presentacion': 'Pieza', 'precio': 2.0, 'cadena': 'Wal-mart', 'marca': 'S/M'},
    {'item': 'Pasta para sopa', 'presentacion': '200 Gr. Fideo', 'precio': 4.0, 'cadena': 'Ley', 'marca': 'Precíssimo'},
    {'item': 'Papel higiénico', 'presentacion': '4 Rollos', 'precio': 7.0, 'cadena': 'Wal-mart', 'marca': 'Lys'},
    {'item': 'Jabón de pasta', 'presentacion': '200 Gr.', 'precio': 9.0, 'cadena': 'Ley', 'marca': 'Zote'},
    {'item': 'Atún', 'presentacion': 'Lata 140 Gr.', 'precio': 10.0, 'cadena': 'Wal-mart', 'marca': 'Fisha Tuna'},
    {'item': 'Leche', 'presentacion': 'Caja 1 Lt. Entera', 'precio': 13.0, 'cadena': 'Wal-mart', 'marca': 'Valley Foods'},
    {'item': 'Tortilla', 'presentacion': '1 Kg. Granel', 'precio': 16.0, 'cadena': 'Ley', 'marca': 'S/M'},
    {'item': 'Arroz', 'presentacion': 'Bolsa 900 Gr.', 'precio': 16.0, 'cadena': 'Ley', 'marca': 'Precíssimo'},
    {'item': 'Frijol', 'presentacion': 'Bolsa 900 Gr.', 'precio': 16.0, 'cadena': 'Ley', 'marca': 'Precíssimo'},
    {'item': 'Azúcar', 'presentacion': 'Bolsa 900 Gr.', 'precio': 20.0, 'cadena': 'Ley', 'marca': 'Ley'},
    {'item': 'Sal', 'presentacion': 'Bote 1 Kg.', 'precio': 24.0, 'cadena': 'Hipermercado Soriana', 'marca': 'Precíssimo'},
    {'item': 'Aceite', 'presentacion': 'Botella 850 Ml.', 'precio': 27.0, 'cadena': 'Ley', 'marca': 'More Value'},
    {'item': 'Pollo', 'presentacion': '1 Kg. Pierna con Muslo', 'precio': 31.0, 'cadena': 'Hipermercado Soriana', 'marca': 'S/M'},
    {'item': 'Huevo', 'presentacion': 'C/18 Blanco', 'precio': 45.0, 'cadena': 'Bodega Aurrera', 'marca': 'Bachoco'},
]

grafica_canasta_personalizada(seleccion_ejemplo, 'Sonora', 'Hermosillo')